# PowerNet: A Learning power allocation in wireless networks

> PowerNet is an MLP-based architecture for learning power allocation in wireless networks. It is designed to be simple and efficient, while still achieving state-of-the-art performance on the problem of power allocation in wireless networks.

In [ ]:
#| default_exp models.power_net

In [ ]:
#| hide
from nbdev.showdoc import *

In [2]:
#| export
import torch
from torch import nn
from torch.nn import functional as F


In [ ]:
#| export
class PowerNetMLP(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, csi_dim, max_power):
        super().__init__()
        self.max_power = max_power
        self.embedding = nn.Embedding(num_embeddings, embedding_dim)
        
        # CSI is complex number → treat as 2 real values per agent
        self.net = nn.Sequential(
            nn.Linear(embedding_dim + csi_dim * 2, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
        )
        self.schedule_head = nn.Linear(64, 1)   # binary send/no send
        self.power_head = nn.Linear(64, 1)       # power level

    def forward(self, message_indices, csi):
        # message_idx: [B] integer indices
        # csi: [B, csi_dim] complex → pass as [B, csi_dim*2] real

        msg_embed = self.embedding(message_indices)      # [B, 49, embedding_dim]
        msg_flat = msg_embed.flatten(1)                  # [B, 49 * embedding_dim]
        csi_real = torch.view_as_real(csi).flatten(-2)   # [B, csi_dim*2]
        x = torch.cat([msg_flat, csi_real], dim=-1)
        features = self.net(x)
        
        schedule = torch.sigmoid(self.schedule_head(features))        # [B, 1] ∈ (0,1)
        power = torch.sigmoid(self.power_head(features)) * self.max_power  # [B, 1] ∈ (0, P_max)
        
        return schedule, power

## Transformer-based PowerNET

In [ ]:
#| export
class PowerNetMasked(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, csi_dim, max_power,
                 nhead=4, num_layers=2):
        super().__init__()
        self.max_power = max_power
        self.embedding_dim = embedding_dim

        # Message token embedding
        self.msg_embedding = nn.Embedding(num_embeddings, embedding_dim)

        # Project CSI (complex → 2 reals) to embedding_dim to use as query token
        self.csi_proj = nn.Linear(csi_dim * 2, embedding_dim)

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=nhead,
            dim_feedforward=256,
            dropout=0.1,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Output heads — operate on the CSI query token output
        self.schedule_head = nn.Linear(embedding_dim, 1)
        self.power_head = nn.Linear(embedding_dim, 1)

    def forward(self, message_indices, csi):
        """
        message_indices: [B, 49] — same message for all neighbors
        csi: [B, num_neighbors, csi_dim] complex — different per neighbor
        returns:
            schedule: [B, num_neighbors, 1]
            power:    [B, num_neighbors, 1]
        """
        B, num_neighbors, _ = csi.shape

        # Embed message tokens: [B, 49, D]
        msg_tokens = self.msg_embedding(message_indices.view(B, -1))

        # Project CSI to embedding dim: [B, num_neighbors, D]
        csi_real = torch.view_as_real(csi).flatten(-2)      # [B, num_neighbors, csi_dim*2]
        csi_tokens = self.csi_proj(csi_real)                 # [B, num_neighbors, D]

        # Process each neighbor separately but efficiently using batch dim
        # Expand message tokens for each neighbor: [B*num_neighbors, 49, D]
        msg_expanded = msg_tokens.unsqueeze(1).expand(
            -1, num_neighbors, -1, -1
        ).reshape(B * num_neighbors, 49, self.embedding_dim)

        # CSI token as first token (acts as query over message)
        # [B*num_neighbors, 1, D]
        csi_expanded = csi_tokens.reshape(B * num_neighbors, 1, self.embedding_dim)

        # Concatenate: CSI token + 49 message tokens → sequence of 50
        sequence = torch.cat([csi_expanded, msg_expanded], dim=1)  # [B*N, 50, D]

        # Transformer: CSI token attends to message tokens
        out = self.transformer(sequence)                     # [B*N, 50, D]

        # Take output at CSI token position (index 0)
        csi_out = out[:, 0, :]                               # [B*N, D]

        # Output heads
        schedule = torch.sigmoid(
            self.schedule_head(csi_out)
        ).view(B, num_neighbors, 1)                          # [B, num_neighbors, 1]

        power = torch.sigmoid(
            self.power_head(csi_out)
        ).view(B, num_neighbors, 1) * self.max_power         # [B, num_neighbors, 1]

        return schedule, power
    

In [ ]:
#| hide
# B=8, 3 neighbors, 1 CSI value per neighbor
B = 8
neighbors= 3
powernet = PowerNetMasked(num_embeddings=256, embedding_dim=256, csi_dim=1, max_power=10.0)
obs = torch.randn(B, 3, 224, 224)  # [B, C, H, W]
message_indices = torch.randint(0, 256, (B, 7, 7))  # [B, 7, 7]
message_flat = message_indices.view(B, -1)                 # [B, 49]
csi = torch.randn(B, neighbors, 1, dtype=torch.complex64)          # [B, 3, 1]

schedule, power = powernet(message_flat, csi)
# schedule: [B, 3, 1] — per neighbor send/no-send
# power:    [B, 3, 1] — per neighbor power level

In [33]:
#| hide
power

tensor([[[3.6058],
         [3.7526],
         [3.4836]],

        [[4.5255],
         [3.5999],
         [3.5742]],

        [[5.5217],
         [3.5087],
         [3.6000]],

        [[4.6323],
         [3.3763],
         [4.6673]],

        [[5.2038],
         [3.3778],
         [3.3321]],

        [[4.3735],
         [3.2546],
         [4.6422]],

        [[4.6176],
         [5.6576],
         [4.2385]],

        [[4.8412],
         [3.2952],
         [3.5338]]], grad_fn=<MulBackward0>)

## Cross-Attention PowerNET

In [34]:
#| export
class PowerNet(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, csi_dim, max_power,
                 nhead=4, num_layers=2):
        super().__init__()
        self.max_power = max_power
        self.embedding_dim = embedding_dim

        self.msg_embedding = nn.Embedding(num_embeddings, embedding_dim)
        self.csi_proj = nn.Linear(csi_dim * 2, embedding_dim)

        # Self-attention over message tokens first
        msg_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=nhead,
            dim_feedforward=256,
            dropout=0.1,
            batch_first=True
        )
        self.msg_transformer = nn.TransformerEncoder(msg_layer, num_layers=num_layers)

        # Cross-attention: CSI queries over message tokens
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=nhead,
            batch_first=True
        )

        self.schedule_head = nn.Linear(embedding_dim, 1)
        self.power_head = nn.Linear(embedding_dim, 1)

    def forward(self, message_indices, csi):
        """
        message_indices: [B, 49]
        csi: [B, num_neighbors, csi_dim] complex
        returns:
            schedule: [B, num_neighbors, 1]
            power:    [B, num_neighbors, 1]
        """
        B, num_neighbors, _ = csi.shape

        # 1. Embed and contextualize message tokens with self-attention
        msg_tokens = self.msg_embedding(message_indices)     # [B, 49, D]
        msg_ctx = self.msg_transformer(msg_tokens)           # [B, 49, D]

        # 2. Project CSI to query vectors
        csi_real = torch.view_as_real(csi).flatten(-2)       # [B, num_neighbors, csi_dim*2]
        csi_query = self.csi_proj(csi_real)                  # [B, num_neighbors, D]

        # 3. Cross-attention: each neighbor's CSI queries over message tokens
        # Expand msg_ctx for each neighbor
        msg_expanded = msg_ctx.unsqueeze(1).expand(
            -1, num_neighbors, -1, -1
        ).reshape(B * num_neighbors, 49, self.embedding_dim) # [B*N, 49, D]

        csi_expanded = csi_query.reshape(
            B * num_neighbors, 1, self.embedding_dim
        )                                                     # [B*N, 1, D]

        # CSI is query, message tokens are keys and values
        # "given my channel quality, what parts of this message matter?"
        attended, _ = self.cross_attn(
            query=csi_expanded,                              # [B*N, 1, D]
            key=msg_expanded,                                # [B*N, 49, D]
            value=msg_expanded                               # [B*N, 49, D]
        )                                                    # [B*N, 1, D]

        out = attended.squeeze(1)                            # [B*N, D]

        # 4. Per-neighbor decisions
        schedule = torch.sigmoid(
            self.schedule_head(out)
        ).view(B, num_neighbors, 1)

        power = torch.sigmoid(
            self.power_head(out)
        ).view(B, num_neighbors, 1) * self.max_power

        return schedule, power

In [35]:
#| hide
# B=8, 3 neighbors, 1 CSI value per neighbor
B = 8
neighbors= 3
powernet = PowerNet(num_embeddings=256, embedding_dim=256, csi_dim=1, max_power=10.0)
obs = torch.randn(B, 3, 224, 224)  # [B, C, H, W]
message_indices = torch.randint(0, 256, (B, 7, 7))  # [B, 7, 7]
message_flat = message_indices.view(B, -1)                 # [B, 49]
csi = torch.randn(B, neighbors, 1, dtype=torch.complex64)          # [B, 3, 1]

schedule, power = powernet(message_flat, csi)
# schedule: [B, 3, 1] — per neighbor send/no-send
# power:    [B, 3, 1] — per neighbor power level

In [38]:
schedule.shape

torch.Size([8, 3, 1])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()